In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

# Quimb imports
import quimb.tensor as qtn

# Qiskit imports
import qiskit as qk

# General imports
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# JAX-like import
import jax

# Custom imports
import mpo_utils
import noise_channels

from lpdo_utils import build_lpdo, lpdo_to_mpo
from training_functions import optimize_tn

seed = 1
rnd_key = jax.random.PRNGKey(seed)
np.random.seed(seed)

def load_results(file_name: str):
    """
    Save results of the function 'optimize_tn' in json format. 
    Complex values are transformed into strings for json serialisation.
    """
    with open(file_name, 'r') as file:
        res = json.load(file)

        res['params'] = [{int(k): np.array(v, dtype=complex) for k, v in p.items()} for p in res['params']]
        
        mpo_params_keys = ['params_true_mpo', 'params_inv_opt_mpo', 'params_inv_true_mpo']

        ignore_keys = ['probs', 'data_file', 'params', 'noise_type', 'lr'] + mpo_params_keys
        for k in set(res.keys()).difference(ignore_keys):
            res[k] = np.real_if_close(np.array(res[k], dtype=complex))
        
    return res

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [2]:
file_name = '../data/optimisation_data/cxeven_nq10_gen_dict_bigger_231127_kyiv_50q_ns10000000_global_repeated_sic_states_2023-11-30_seed108.json'
res = load_results(file_name=file_name)

# Find parameters attaining lowest test error
best_idx = np.argmin(res['test_loss'])
print(f"Min Test Loss = {np.min(res['test_loss'])} at epoch {best_idx}")
opt_params = res['params'][best_idx]
num_qubits = len(opt_params)

# Create LPDO and MPO (in PTM form) with such optimal parametes
opt_lpdo = build_lpdo(tn_data = opt_params.values())   
opt_mpo = lpdo_to_mpo(opt_lpdo)

# Create true LPDO and MPO (in PTM form) for the corresponding noise
noise_type = res['noise_type']
true_lpdo = noise_channels.select_noise_channel(noise_type, num_qubits, tensor_type = 'lpdo')
true_mpo  = noise_channels.select_noise_channel(noise_type, num_qubits, tensor_type = 'mpo')
#mpo_utils.mpo_to_params_file(true_mpo, file_name, save_key = 'params_true_mpo')

print(f"‖ LPDO_opt - LPDO_true ‖_Frob = {(opt_lpdo.distance(true_lpdo) / 2**num_qubits) ** 2}") 
print(f"‖ MPO_opt - MPO_true ‖_Frob = {(opt_mpo.distance(true_mpo) / 2**num_qubits) ** 2}") 

Min Test Loss = 5.500145158605074 at epoch 3
‖ LPDO_opt - LPDO_true ‖_Frob = 0.00010987176124976904
‖ MPO_opt - MPO_true ‖_Frob = 0.0001098717612485478


In [7]:
#######################
# EXPLICIT INVERSION
#######################
num_sweeps = 1
bond_dim = 8 # * opt_mpo_compressed.ind_size(opt_mpo.inner_inds()[0])
inv_opt_mpo, _ = mpo_utils.invert_mpo(opt_mpo, bond_dim, num_sweeps, canonize_while_sweeping=False, dtype=float, imag_tol = 1e-12)

Initial distance between B and A^-1: ‖ B * A - Id ‖ _Frob =  1024.000438634358
Iteration 1, qubit 0, ||A*B-I||_F = 1023.9999999928681
Iteration 1, qubit 1, ||A*B-I||_F = 1023.9999998091387
Iteration 1, qubit 2, ||A*B-I||_F = 1023.9999961798579
Iteration 1, qubit 3, ||A*B-I||_F = 1023.9999428284466
Iteration 1, qubit 4, ||A*B-I||_F = 1023.9973814574441
Iteration 1, qubit 5, ||A*B-I||_F = 1023.978631344096
Iteration 1, qubit 6, ||A*B-I||_F = 1023.4850919878179
Iteration 1, qubit 7, ||A*B-I||_F = 1007.0626881680031
Iteration 1, qubit 8, ||A*B-I||_F = 633.6352812991215
Iteration 1, qubit 9, ||A*B-I||_F = 5.159303863488407


In [5]:
#######################
# EXPLICIT INVERSION
#######################
num_sweeps = 1
bond_dim = 4 # * opt_mpo_compressed.ind_size(opt_mpo.inner_inds()[0])
inv_opt_mpo, _ = mpo_utils.invert_mpo(opt_mpo, bond_dim, num_sweeps, canonize_while_sweeping=True, dtype=float, imag_tol = 1e-12)

#######################
# VARIATIONAL INVERSION
#######################
num_sweeps = 3

inv_opt_mpo = qtn.MatrixProductOperator(inv_opt_mpo.arrays, upper_ind_id='k{}', lower_ind_id='b{}', site_tag_id='I{}')

inv_opt_mpo =  mpo_utils.variational_inversion_mpo(opt_mpo, virtual_bd = bond_dim, warmstart_mpo = None, 
                                                   optimisation_method='global', optimizer='L-BFGS-B', steps = 5000, num_sweeps=num_sweeps)

inv_opt_mpo =  mpo_utils.variational_inversion_mpo(opt_mpo, virtual_bd = bond_dim, warmstart_mpo = None, 
                                                   optimisation_method='local', optimizer='L-BFGS-B', steps = 1000, num_sweeps=num_sweeps)

inv_opt_mpo =  mpo_utils.variational_inversion_mpo(opt_mpo, virtual_bd = bond_dim, warmstart_mpo = None, 
                                                   optimisation_method='global', optimizer='L-BFGS-B', steps = 1000, num_sweeps=num_sweeps)

#mpo_utils.mpo_to_params_file(inv_opt_mpo, file_name, save_key = 'params_inv_opt_mpo')

Initial distance between B and A^-1: ‖ B * A - Id ‖ _Frob =  1024.0004399612758
Iteration 1, qubit 0, ||A*B-I||_F = 1023.9999999869382
Iteration 1, qubit 1, ||A*B-I||_F = 1023.9999994713587
Iteration 1, qubit 2, ||A*B-I||_F = 1023.9999937169696
Iteration 1, qubit 3, ||A*B-I||_F = 1023.999946076622
Iteration 1, qubit 4, ||A*B-I||_F = 1023.9965229548523
Iteration 1, qubit 5, ||A*B-I||_F = 1023.9666426566736
Iteration 1, qubit 6, ||A*B-I||_F = 1023.6102158403902
Iteration 1, qubit 7, ||A*B-I||_F = 1017.40998295027
Iteration 1, qubit 8, ||A*B-I||_F = 935.2289212482798
Iteration 1, qubit 9, ||A*B-I||_F = 6.409121305769633
Before optimisation:
 > ‖ Λ(θ) - Λ_targ‖ =  972.5508417904824
 > ‖ Λ(θ) - Id ‖ =  1024.0004881826696
 > ‖ Λ(θ) * Λ_targ  - Id ‖ =  1024.000440520033
 


+0.853057107136 [best: +0.853057107136] : : 5339it [02:11, 40.49it/s]                                                             


Before optimisation:
 > ‖ Λ(θ) - Λ_targ‖ =  972.5508418270321
 > ‖ Λ(θ) - Id ‖ =  1024.0004882141648
 > ‖ Λ(θ) * Λ_targ  - Id ‖ =  1024.0004396267207
 
Sweep 0 —— DIRECT SWEEP
Optimising local tensor O0: 

+1023.999999999672 [best: +1023.999999999672] :   1%|          | 8/1000 [00:00<01:14, 13.40it/s]


Optimising local tensor O1: 

...

+1.985823743601 [best: +1.985823743601] :  27%|██▋       | 270/1000 [00:04<00:12, 56.23it/s]


Optimising local tensor O3: 

+1.977980759148 [best: +1.977980759148] : : 1073it [00:13, 78.02it/s]                        


Optimising local tensor O4: 

+1.474056353826 [best: +1.474056353826] :   8%|▊         | 83/1000 [00:01<00:12, 75.63it/s]  

In [4]:
dir(inv_opt_mpo)

['H',
 'L',
 '_CONTRACT_STRUCTURED',
 '_EXTRA_PROPS',
 '_L',
 '_NDIMS',
 '__add__',
 '__and__',
 '__class__',
 '__copy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__irshift__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__lt__',
 '__matmul__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__rshift__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__truediv__',
 '__weakref__',
 '__xor__',
 '_apply_mpo',
 '_apply_mps',
 '_canonize_around_tids',
 '_canonize_between_tids',
 '_compatible_1d',
 '_compatible_arbgeom',
 '_compress_between_full_bond_tids',
 '_compress_between_local_fit',
 '_compress_bet

In [7]:
#######################
# EXPLICIT INVERSION
#######################
bond_dim = true_mpo.ind_size(true_mpo.inner_inds()[0])
print(bond_dim)

num_sweeps = 2
inv_true_mpo, e2 = mpo_utils.invert_mpo(true_mpo, bond_dim, num_sweeps, dtype=float)
inv_true_mpo = qtn.MatrixProductOperator(inv_true_mpo.arrays, upper_ind_id='k{}', lower_ind_id='b{}', site_tag_id='I{}')

#######################
# VARIATIONAL INVERSION
#######################
if False:
    num_sweeps = 3

    inv_true_mpo = qtn.MatrixProductOperator(inv_true_mpo.arrays, upper_ind_id='k{}', lower_ind_id='b{}', site_tag_id='I{}')

    inv_true_mpo =  mpo_utils.variational_inversion_mpo(true_mpo, virtual_bd = bond_dim, warmstart_mpo = inv_true_mpo, 
                                                        optimisation_method='global', optimizer='L-BFGS-B', steps = 5000, num_sweeps=num_sweeps)

    inv_true_mpo =  mpo_utils.variational_inversion_mpo(true_mpo, virtual_bd = bond_dim, warmstart_mpo = inv_true_mpo, 
                                                        optimisation_method='local', optimizer='L-BFGS-B', steps = 1000, num_sweeps=num_sweeps)

    inv_true_mpo =  mpo_utils.variational_inversion_mpo(true_mpo, virtual_bd = bond_dim, warmstart_mpo = inv_true_mpo, 
                                                        optimisation_method='global', optimizer='L-BFGS-B', steps = 1000, num_sweeps=num_sweeps)

#mpo_utils.mpo_to_params_file(inv_true_mpo, file_name, save_key = 'params_inv_true_mpo')

4
Initial distance between B and A^-1: ‖ B * A - Id ‖ _Frob =  1024.0004424164024
Iteration 1, qubit 0, ||A*B-I||_F = 1023.999999999573
Iteration 1, qubit 1, ||A*B-I||_F = 1023.9999999951768
Iteration 1, qubit 2, ||A*B-I||_F = 1023.9999995417667
Iteration 1, qubit 3, ||A*B-I||_F = 1023.9999645528864
Iteration 1, qubit 4, ||A*B-I||_F = 1023.9993222815723
Iteration 1, qubit 5, ||A*B-I||_F = 1023.981791328992
Iteration 1, qubit 6, ||A*B-I||_F = 1023.7317410102578
Iteration 1, qubit 7, ||A*B-I||_F = 1019.7490673815586
Iteration 1, qubit 8, ||A*B-I||_F = 932.8164728025267
Iteration 1, qubit 9, ||A*B-I||_F = 1.52587890625e-05
Iteration 2, qubit 8, ||A*B-I||_F = 2.1579186437577746e-05
Iteration 2, qubit 7, ||A*B-I||_F = 0.0
Iteration 2, qubit 6, ||A*B-I||_F = 1.8688123647942946e-05
Iteration 2, qubit 5, ||A*B-I||_F = (9.343313592127634e-22+1.52587890625e-05j)
Iteration 2, qubit 4, ||A*B-I||_F = (1.3213440799491782e-21+2.1579186437577746e-05j)
Iteration 2, qubit 3, ||A*B-I||_F = (1.61830938526